# SQL com Python — Cheat Sheet 🗄️

Neste notebook, exploramos como executar comandos SQL diretamente no Python usando `sqlite3` e `pandas`, conectando-se ao nosso banco de dados local `cheatsheet.db`.

In [ ]:
import sqlite3
import pandas as pd

# Conectando ao banco de dados gerado previamente (Execute create_db.py se não existir)
conn = sqlite3.connect('../datasets/cheatsheet.db')
print("Conectado ao SQLite!")

### 1. Operações Básicas (Filtros e Selects)

In [ ]:
query = """
SELECT 
    nome, departamento, salario 
FROM employees 
WHERE salario > 8000 
ORDER BY salario DESC 
LIMIT 5;
"""
pd.read_sql(query, conn)

### 2. Agregações (GROUP BY, SUM, AVG)

In [ ]:
query = """
SELECT 
    departamento,
    COUNT(*) AS qtd_funcionarios,
    ROUND(AVG(salario), 2) AS media_salarial,
    SUM(salario) AS custo_total
FROM employees
GROUP BY departamento
ORDER BY custo_total DESC;
"""
pd.read_sql(query, conn)

### 3. Joins (Múltiplas Tabelas)

In [ ]:
query = """
SELECT 
    c.nome AS cliente,
    c.segmento,
    COUNT(o.id_pedido) AS total_pedidos,
    SUM(o.valor_total) AS gasto_total
FROM customers c
LEFT JOIN orders o ON c.id_cliente = o.id_cliente
GROUP BY c.id_cliente, c.nome, c.segmento
ORDER BY gasto_total DESC
LIMIT 5;
"""
pd.read_sql(query, conn)

### 4. Window Functions (Rank, Participação %)

In [ ]:
query = """
SELECT 
    nome, 
    departamento, 
    salario,
    RANK() OVER(PARTITION BY departamento ORDER BY salario DESC) AS rank_depto,
    ROUND((salario * 100.0) / SUM(salario) OVER(PARTITION BY departamento), 1) AS pct_do_depto
FROM employees
WHERE salario IS NOT NULL
LIMIT 10;
"""
pd.read_sql(query, conn)

### Alternativa: DuckDB (Sem Banco de Dados Pré-Criado)
O `duckdb` permite rodar SQL diretamente nos arquivos CSV, sem precisar popular um `.db` antes.

In [ ]:
# Descomente e instale se necessário: !pip install duckdb
import duckdb

query_duck = """
SELECT departamento, COUNT(*) as qtd
FROM read_csv_auto('../datasets/employees.csv')
GROUP BY departamento
ORDER BY qtd DESC;
"""
duckdb.query(query_duck).df()

In [ ]:
# Fechando a conexão SQLite ao final
conn.close()